In [1]:
import torch
import torch.nn as nn
from transformers import CLIPProcessor, CLIPModel, AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm
import random

/data/2_data_server/cv-07/anaconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- 1. Configuration and Path Definitions ---
print("Setting up configurations and paths...")

# Model names (must comply with competition rules: pre-2024, open-source, total params < 3B)
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14" # ~0.3B parameters
GPT2_MODEL_NAME = "gpt2-xl"                       # ~1.558B parameters
# Total base parameters: 0.3B (CLIP) + 1.558B (GPT-2 XL) = 1.858B.
# This leaves ~1.142B for the trainable mapping network.

# Absolute path to your preprocessed VMCBench dataset
VMCBENCH_BASE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/clip_gpt/VMCBench_from_CSV_as_Official_VQAv2"

# Training data paths
TRAIN_ANNOTATION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_mscoco_train2014_annotations.json")
TRAIN_QUESTION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_train2014_questions.json")
TRAIN_IMAGE_DIR = os.path.join(VMCBENCH_BASE_DIR, "train2014")

# Validation data paths (highly recommended for monitoring training progress)
VAL_ANNOTATION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_mscoco_val2014_annotations.json")
VAL_QUESTION_FILE = os.path.join(VMCBENCH_BASE_DIR, "vqa", "v2_OpenEnded_mscoco_val2014_questions.json")
VAL_IMAGE_DIR = os.path.join(VMCBENCH_BASE_DIR, "val2014")

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Setting up configurations and paths...
Using device: cuda


In [3]:
# --- 2. Load Models and Tokenizer ---
print("Loading models and tokenizer...")

# Load CLIP model and freeze its weights (not trainable)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
clip_model.eval() # Set to evaluation mode
for param in clip_model.parameters():
    param.requires_grad = False # Freeze parameters
print(f"CLIP model loaded. Parameters: {sum(p.numel() for p in clip_model.parameters()) / 1e9:.3f}B")

# Load GPT-2 model and freeze its weights (not trainable)
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL_NAME)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL_NAME).to(device)
gpt2_model.eval() # Set to evaluation mode
for param in gpt2_model.parameters():
    param.requires_grad = False # Freeze parameters

# Adjust GPT-2 tokenizer: use EOS token as pad token for consistent padding
if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id
print(f"GPT-2 model loaded. Parameters: {sum(p.numel() for p in gpt2_model.parameters()) / 1e9:.3f}B")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading models and tokenizer...
CLIP model loaded. Parameters: 0.428B
GPT-2 model loaded. Parameters: 1.558B


In [4]:
# --- 3. Define Mapping Network ---
# This network transforms CLIP image embeddings into GPT-2 soft prompt embeddings.
class ImageToSoftPromptMapping(nn.Module):
    def __init__(self, clip_embedding_dim, gpt2_hidden_dim, num_soft_tokens):
        super().__init__()
        self.num_soft_tokens = num_soft_tokens
        self.gpt2_hidden_dim = gpt2_hidden_dim # Store as instance variable
        # A simple MLP architecture. You can experiment with more layers or different activations.
        self.mlp = nn.Sequential(
            nn.Linear(clip_embedding_dim, gpt2_hidden_dim * num_soft_tokens // 2),
            nn.ReLU(),
            nn.Linear(gpt2_hidden_dim * num_soft_tokens // 2, gpt2_hidden_dim * num_soft_tokens)
        )

    def forward(self, image_features):
        mapped_features = self.mlp(image_features)
        # Reshape to (batch_size, num_soft_tokens, gpt2_hidden_dim) to match GPT-2's embedding input format
        soft_prompt_embeddings = mapped_features.view(-1, self.num_soft_tokens, self.gpt2_hidden_dim)
        return soft_prompt_embeddings

# Define dimensions based on loaded models
CLIP_EMBEDDING_DIM = clip_model.config.projection_dim # e.g., 768 for ViT-L/14, 512 for ViT-B/32
GPT2_HIDDEN_DIM = gpt2_model.config.hidden_size       # e.g., 1600 for GPT-2 XL

mapping_network = ImageToSoftPromptMapping(CLIP_EMBEDDING_DIM, GPT2_HIDDEN_DIM, NUM_SOFT_TOKENS).to(device)

# Verify trainable parameters of the mapping network (must fit within 3B total limit)
total_mapping_params = sum(p.numel() for p in mapping_network.parameters() if p.requires_grad)
print(f"Mapping Network Trainable Parameters: {total_mapping_params / 1e6:.2f}M")
print(f"Total Model Parameters (CLIP + GPT-2 + Mapping): {(sum(p.numel() for p in clip_model.parameters()) + sum(p.numel() for p in gpt2_model.parameters()) + total_mapping_params) / 1e9:.3f}B")


Mapping Network Trainable Parameters: 524.34M
Total Model Parameters (CLIP + GPT-2 + Mapping): 2.510B


In [5]:
# --- 4. Custom Dataset Class for VMCBench ---
class VMCBenchDataset(Dataset):
    def __init__(self, annotation_file, question_file, image_dir, tokenizer, clip_processor, split_type):
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.clip_processor = clip_processor
        self.split_type = split_type # e.g., "train2014" or "val2014"

        with open(annotation_file, 'r', encoding='utf-8') as f:
            self.annotations = json.load(f)["annotations"]
        with open(question_file, 'r', encoding='utf-8') as f:
            self.questions_data = {q["question_id"]: q for q in json.load(f)["questions"]}

        # Filter annotations to ensure corresponding question data exists
        self.annotations = [anno for anno in self.annotations if anno["question_id"] in self.questions_data]

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        anno = self.annotations[idx]
        question_id = anno["question_id"]
        image_id = anno["image_id"]
        
        question_info = self.questions_data[question_id]
        # question_text already includes choices (A, B, C, D) from your preprocessing script
        question_text = question_info["question"] 

        # Load image
        image_filename = f"COCO_{self.split_type}_{str(image_id).zfill(12)}.jpg"
        image_path = os.path.join(self.image_dir, image_filename)
        try:
            image = Image.open(image_path).convert("RGB")
        except FileNotFoundError:
            print(f"Warning: Image file not found: {image_path}. Returning None for this item.")
            return None # Return None if image is missing, will be filtered in collate_fn
            
        # CLIP image preprocessing
        clip_image_inputs = self.clip_processor(images=image, return_tensors="pt")["pixel_values"].squeeze(0)

        # Construct GPT-2 input prompt: "Question + Choices + Answer:"
        full_prompt = f"{question_text}\nAnswer:"
        # Tokenize without padding or explicit truncation here; collate_fn will handle it
        gpt2_inputs = self.tokenizer(full_prompt, return_tensors="pt", padding=False, truncation=True, max_length=512)

        # Get the target answer token ID (e.g., token ID for 'A', 'B', 'C', or 'D')
        target_char_label = chr(ord('A') + anno["answer_label"]) # Convert 0->'A', 1->'B', etc.
        target_ids = self.tokenizer(target_char_label, return_tensors="pt")["input_ids"].squeeze(0)
        
        # Combine GPT-2's original input IDs with the target token ID for collate_fn
        # This combined tensor helps in identifying the target token later in collate_fn
        labels_for_collate_item = torch.cat([gpt2_inputs["input_ids"].squeeze(0), target_ids], dim=0)

        return {
            "clip_image_inputs": clip_image_inputs,
            "gpt2_input_ids_for_collate": gpt2_inputs["input_ids"].squeeze(0),
            "attention_mask_for_collate": gpt2_inputs["attention_mask"].squeeze(0),
            "labels_for_collate": labels_for_collate_item
        }


In [6]:
# --- 5. Custom Collate Function ---
def collate_fn(batch):
    # Filter out any None items (e.g., from missing images)
    batch = [item for item in batch if item is not None]
    if not batch: # If batch becomes empty after filtering, return None
        return None 
    
    # Stack CLIP image inputs (all images processed to same size by processor)
    clip_image_inputs = torch.stack([item['clip_image_inputs'] for item in batch])
    
    # Calculate the maximum length of the text prompt part (Question + Choices + Answer:) in the batch
    max_len_text_input = max(item['gpt2_input_ids_for_collate'].size(0) for item in batch)
    
    # Determine the final sequence length for GPT-2's inputs_embeds and labels
    # It's NUM_SOFT_TOKENS (for soft prompt) + max_len_text_input (for padded text prompt)
    max_seq_len_for_inputs_and_labels = NUM_SOFT_TOKENS + max_len_text_input 

    padded_gpt2_input_ids = []
    padded_attention_mask = []
    padded_labels = []

    for item in batch:
        gpt2_input_ids_unpadded = item['gpt2_input_ids_for_collate'] # Original unpadded input_ids for current item
        attention_mask_unpadded = item['attention_mask_for_collate']
        target_token_id = item['labels_for_collate'][-1].item() # Extract the single target token ID

        # 1. Pad GPT-2 text input IDs to `max_len_text_input`
        pad_len_input = max_len_text_input - gpt2_input_ids_unpadded.size(0)
        padded_gpt2_input_ids_item = torch.cat([gpt2_input_ids_unpadded, torch.full((pad_len_input,), gpt2_tokenizer.pad_token_id, dtype=torch.long)], dim=0)
        padded_gpt2_input_ids.append(padded_gpt2_input_ids_item)
        
        # 2. Pad attention mask for GPT-2 text input
        padded_attention_mask_item = torch.cat([attention_mask_unpadded, torch.zeros(pad_len_input, dtype=torch.long)], dim=0)
        padded_attention_mask.append(padded_attention_mask_item)
        
        # 3. Construct labels for the *entire extended sequence*.
        # Initialize `current_item_labels` with -100 for all positions (ignore loss).
        current_item_labels = torch.full((max_seq_len_for_inputs_and_labels,), -100, dtype=torch.long)
        
        # Determine the position where the answer token should be predicted.
        # This is the index of the *last token of the prompt text* (e.g., "Answer:") within the extended input sequence.
        # GPT-2 will predict the *next* token based on this one.
        # The index is `NUM_SOFT_TOKENS` (for soft tokens) + `(length of this item's unpadded text prompt) - 1`.
        answer_label_position = NUM_SOFT_TOKENS + gpt2_input_ids_unpadded.size(0) - 1

        # Place the `target_token_id` at this specific `answer_label_position`.
        # Add defensive checks for index bounds.
        if answer_label_position >= 0 and \
           answer_label_position < max_seq_len_for_inputs_and_labels:
            current_item_labels[answer_label_position] = target_token_id
        else:
            # This warning means the calculated position is out of bounds, which indicates a problem.
            print(f"CRITICAL WARNING: Answer label position ({answer_label_position}) is out of bounds for labels length ({max_seq_len_for_inputs_and_labels}). This item's label might be incorrect. Prompt length: {gpt2_input_ids_unpadded.size(0)}, Soft tokens: {NUM_SOFT_TOKENS}, Max Text Input: {max_len_text_input}, Target ID: {target_token_id}")

        padded_labels.append(current_item_labels)

    return {
        "clip_image_inputs": clip_image_inputs,
        "gpt2_input_ids": torch.stack(padded_gpt2_input_ids), # This is the padded text portion of input_ids
        "attention_mask": torch.stack(padded_attention_mask),
        "labels": torch.stack(padded_labels)
    }


In [7]:
# --- 6. DataLoader Creation ---
print("Creating DataLoaders...")
train_dataset = VMCBenchDataset(
    annotation_file=TRAIN_ANNOTATION_FILE,
    question_file=TRAIN_QUESTION_FILE,
    image_dir=TRAIN_IMAGE_DIR,
    tokenizer=gpt2_tokenizer,
    clip_processor=clip_processor,
    split_type="train2014"
)
# Note: If collate_fn returns None for an empty batch, the DataLoader might raise an error.
# Ensure your dataset is robust or add more robust error handling around DataLoader iteration.
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Number of train batches: {len(train_dataloader)}")

# Validation DataLoader (highly recommended for monitoring overfitting)
val_dataset = VMCBenchDataset(
    annotation_file=VAL_ANNOTATION_FILE,
    question_file=VAL_QUESTION_FILE,
    image_dir=VAL_IMAGE_DIR,
    tokenizer=gpt2_tokenizer,
    clip_processor=clip_processor,
    split_type="val2014"
)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Number of validation batches: {len(val_dataloader)}")

Creating DataLoaders...
Train dataset size: 1000
Number of train batches: 500
Validation dataset size: 60
Number of validation batches: 30


In [8]:
# Hyperparameters
BATCH_SIZE = 4 # Adjust based on your GPU memory. GPT-2 XL is very large.
NUM_EPOCHS = 10
LEARNING_RATE = 1e-5
NUM_SOFT_TOKENS = 20 # Number of virtual tokens to prepend to GPT-2 input. Experiment with values like 10-50.

# --- 7. Training Loop ---
print("Starting training of the mapping network...")
optimizer = torch.optim.AdamW(mapping_network.parameters(), lr=LEARNING_RATE) # Only training mapping network
scaler = torch.cuda.amp.GradScaler() # For mixed precision training (recommended for GPUs)

best_val_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    mapping_network.train() # Set mapping network to training mode
    total_train_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} (Train)")

    for batch in progress_bar:
        # Skip batch if collate_fn returned None (e.g., due to missing images)
        if batch is None:
            continue

        optimizer.zero_grad()

        # Move batch data to the correct device
        clip_image_inputs = batch["clip_image_inputs"].to(device)
        gpt2_input_ids = batch["gpt2_input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Get CLIP image features (CLIP is frozen, so no_grad is efficient)
        with torch.no_grad():
            image_features = clip_model.get_image_features(pixel_values=clip_image_inputs)
        
        # Generate soft prompt embeddings using the trainable mapping network
        soft_prompt_embeddings = mapping_network(image_features)

        # Get GPT-2 text input embeddings
        # This accesses GPT-2's internal word token embeddings (wte) layer
        input_embeddings = gpt2_model.get_input_embeddings()(gpt2_input_ids)
        
        # Concatenate soft prompt embeddings with text input embeddings
        # Soft prompts are prepended to the text input
        extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
        
        # Extend attention mask for soft prompts
        soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
        extended_attention_mask = torch.cat((soft_prompt_attention_mask, attention_mask), dim=1)
        
        # Perform forward pass with mixed precision (if using GPU)
        with torch.cuda.amp.autocast():
            outputs = gpt2_model(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                labels=labels # GPT-2 will compute loss with these labels
            )

            loss = outputs.loss

        # Backward pass and optimization
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_train_loss += loss.item()
        progress_bar.set_postfix(loss=total_train_loss / (progress_bar.n + 1))

    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}")

    # --- Validation Loop ---
    mapping_network.eval() # Set mapping network to evaluation mode
    total_val_loss = 0
    val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} (Validation)")
    with torch.no_grad(): # No gradients needed for validation
        for batch in val_progress_bar:
            if batch is None:
                continue

            clip_image_inputs = batch["clip_image_inputs"].to(device)
            gpt2_input_ids = batch["gpt2_input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            image_features = clip_model.get_image_features(pixel_values=clip_image_inputs)
            soft_prompt_embeddings = mapping_network(image_features)
            input_embeddings = gpt2_model.get_input_embeddings()(gpt2_input_ids)
            extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
            
            soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
            extended_attention_mask = torch.cat((soft_prompt_attention_mask, attention_mask), dim=1)

            outputs = gpt2_model(
                inputs_embeds=extended_input_embeddings,
                attention_mask=extended_attention_mask,
                labels=labels
            )
            loss = outputs.loss
            total_val_loss += loss.item()
            val_progress_bar.set_postfix(loss=total_val_loss / (val_progress_bar.n + 1))

    avg_val_loss = total_val_loss / len(val_dataloader)
    print(f"Epoch {epoch+1} Validation Loss: {avg_val_loss:.4f}")

    # Save the best model based on validation loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        mapping_network_save_path = "best_mapping_network_weights.pth"
        torch.save(mapping_network.state_dict(), mapping_network_save_path)
        print(f"Validation loss improved! Model saved to: {mapping_network_save_path}")

print("Mapping network training complete.")

/tmp/ipykernel_3600527/1106855792.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For mixed precision training (recommended for GPUs)


Starting training of the mapping network...


Epoch 1/5 (Train):   0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipykernel_3600527/1106855792.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
Epoch 1/5 (Train): 100%|██████████| 500/500 [01:25<00:00,  5.85it/s, loss=1.84]


Epoch 1 Train Loss: 1.8407


Epoch 1/5 (Validation): 100%|██████████| 30/30 [00:02<00:00, 10.13it/s, loss=1.5] 


...

Epoch 4 Validation Loss: 2.5343


Epoch 5/5 (Train): 100%|██████████| 500/500 [01:24<00:00,  5.95it/s, loss=0.377]


Epoch 5 Train Loss: 0.3765


Epoch 5/5 (Validation): 100%|██████████| 30/30 [00:02<00:00, 10.84it/s, loss=2.24]

Epoch 5 Validation Loss: 2.1693
Mapping network training complete.


In [13]:
def predict_answer(image_path, question, choices):
    # 1. 이미지 로드 및 CLIP 임베딩 추출
    image = Image.open(image_path).convert("RGB")
    clip_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    
    with torch.no_grad():
        image_features = clip_model.get_image_features(pixel_values=clip_inputs["pixel_values"])
    
    # 2. 매핑 네트워크를 통해 소프트 프롬프트 임베딩 생성
    with torch.no_grad():
        soft_prompt_embeddings = loaded_mapping_network(image_features)

    # 3. GPT-2 프롬프트 구성 (질문 + 보기)
    choices_text = [
        f"A. {choices[0]}",
        f"B. {choices[1]}",
        f"C. {choices[2]}",
        f"D. {choices[3]}"
    ]
    
    joined_choices_text = '\n'.join(choices_text)
    full_question_text = f"{question}\n{joined_choices_text}\nAnswer:"

    gpt2_text_inputs = gpt2_tokenizer(full_question_text, return_tensors="pt").to(device)

    # 4. GPT-2 텍스트 입력의 임베딩 가져오기
    input_embeddings = gpt2_model.get_input_embeddings()(gpt2_text_inputs["input_ids"])

    # 5. 소프트 프롬프트와 텍스트 임베딩 결합
    extended_input_embeddings = torch.cat((soft_prompt_embeddings, input_embeddings), dim=1)
    
    # 6. 어텐션 마스크 확장
    soft_prompt_attention_mask = torch.ones(soft_prompt_embeddings.shape[0], soft_prompt_embeddings.shape[1], dtype=torch.long).to(device)
    extended_attention_mask = torch.cat((soft_prompt_attention_mask, gpt2_text_inputs["attention_mask"]), dim=1)

    # 7. GPT-2 답변 생성 (다음 토큰을 예측하도록)
    with torch.no_grad():
        output_ids = gpt2_model.generate(
            inputs_embeds=extended_input_embeddings,
            attention_mask=extended_attention_mask,
            max_new_tokens=1, # 오직 하나의 정답 문자만 생성
            num_beams=1,      # 빔 서치 사용 안함
            do_sample=False,  # 샘플링 사용 안함 (가장 높은 확률 토큰 선택)
            pad_token_id=gpt2_tokenizer.pad_token_id,
            eos_token_id=gpt2_tokenizer.eos_token_id # EOS 토큰 만나면 생성 중단
        )
    
    # 8. 생성된 토큰에서 원래 프롬프트 부분 제거 후 디코딩
    # `output_ids`는 이제 대부분의 경우 새로 생성된 토큰만을 포함합니다.
    # 따라서 output_ids의 마지막 토큰이 우리가 원하는 예측 토큰입니다.
    # `max_new_tokens=1`이므로, 생성된 토큰은 하나 뿐입니다.
    generated_token_id = output_ids[0, -1] # 배치 내 첫 번째 시퀀스(0)의 마지막 토큰(-1) 접근
    
    predicted_char = gpt2_tokenizer.decode(generated_token_id, skip_special_tokens=True).strip()

    # 9. 결과가 'A', 'B', 'C', 'D' 중 하나가 아닐 경우 처리 (Fallback)
    if predicted_char not in ['A', 'B', 'C', 'D']:
        print(f"경고: 예상치 못한 답변 생성: '{predicted_char}'. 랜덤 선택으로 대체합니다.")
        return random.choice(['A', 'B', 'C', 'D'])

    return predicted_char

In [14]:
# --- 9. Inference on Competition Test Dataset ---
# Paths to the competition's test data
TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test.csv"
TEST_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test_input_images"

df_test_input = pd.read_csv(TEST_CSV_PATH)
df_submission = pd.DataFrame(columns=["ID", "answer"])

print("\nStarting inference on the competition test dataset...")
for index, row in tqdm(df_test_input.iterrows(), total=len(df_test_input), desc="Generating predictions"):
    img_id = row['ID']
    # Ensure correct image path: os.path.basename is needed if img_path in CSV is relative
    img_path = os.path.join(TEST_IMAGE_DIR, os.path.basename(row['img_path']))
    question = row['Question']
    choices = [row['A'], row['B'], row['C'], row['D']]

    predicted_answer = predict_answer(img_path, question, choices)
    
    df_submission = pd.concat([df_submission, pd.DataFrame([{"ID": img_id, "answer": predicted_answer}])], ignore_index=True)

# Save the final submission file
SUBMISSION_FILE_PATH = "submission.csv"
df_submission.to_csv(SUBMISSION_FILE_PATH, index=False)
print(f"\nFinal submission file saved to: {SUBMISSION_FILE_PATH}")


Starting inference on the competition test dataset...


Generating predictions:   8%|▊         | 68/852 [00:04<00:52, 14.93it/s]

경고: 예상치 못한 답변 생성: ''. 랜덤 선택으로 대체합니다.


Generating predictions:   9%|▉         | 78/852 [00:05<00:53, 14.58it/s]

경고: 예상치 못한 답변 생성: ''. 랜덤 선택으로 대체합니다.


...

경고: 예상치 못한 답변 생성: ''. 랜덤 선택으로 대체합니다.


Generating predictions:  96%|█████████▌| 814/852 [00:53<00:02, 15.32it/s]

경고: 예상치 못한 답변 생성: ''. 랜덤 선택으로 대체합니다.


Generating predictions: 100%|██████████| 852/852 [00:56<00:00, 15.08it/s]


Final submission file saved to: submission.csv
